In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, StackingClassifier

In [52]:
train = pd.read_csv("train.csv")
X = train.drop(columns=["class"])
y = train["class"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


from sklearn.model_selection import cross_validate

from sklearn.model_selection import cross_validate

def evaluate(model, X, y):

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )

    train_mean = scores["train_score"].mean()
    val_mean = scores["test_score"].mean()
    val_std = scores["test_score"].std()

    return val_mean, val_std, train_mean


Los 4 parámetros del BaggingClassifier:

- max_depth — profundidad del árbol base, controla cuánto aprende cada modelo individual
- n_estimators — cuántos árboles entrena el bagging (50, 100, 200 → óptimo 100)
- max_samples — qué porcentaje del dataset ve cada árbol (0.5–0.8 → óptimo 0.65)
- max_features — qué porcentaje de features usa cada árbol (0.05–0.5 → óptimo 0.1)



### DecisionTree

In [66]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_fix = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tree_base = DecisionTreeClassifier(random_state=42)
scores = cross_validate(tree_base, X, y, cv=cv_fix, scoring="accuracy", return_train_score=True)
print(f"Decision Tree sin bagging → Train: {scores['train_score'].mean():.4f} | CV: {scores['test_score'].mean():.4f} ± {scores['test_score'].std():.4f}")

Decision Tree sin bagging → Train: 1.0000 | CV: 0.6540 ± 0.0364


Estamos explorando el efecto de nestimators y maxsamples en un BaggingClassifier con árbol de decisión sin restricciones, usando todas las features (max_featur…Estamos explorando el efecto de n_estimators y max_samples en un BaggingClassifier con árbol de decisión sin restricciones, usando todas las features (max_features=1.0). El objetivo es entender cómo el número de modelos y el tamaño de cada muestra afectan al rendimiento antes de introducir aleatoriedad en las features.

In [53]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

for n_estimators in [50, 100, 200]:
    for max_samples in [0.5, 0.7, 1.0]:
        bag = BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=1.0,  # usa todas las features (diferencia con RF)
            random_state=42,
            n_jobs=-1
        )
        mean, std, train = evaluate(bag, X, y)
        print(f"n={n_estimators} samples={max_samples} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

n=50 samples=0.5 → Train: 0.9778 | CV: 0.7840 ± 0.0201
n=50 samples=0.7 → Train: 0.9970 | CV: 0.7990 ± 0.0389
n=50 samples=1.0 → Train: 1.0000 | CV: 0.7980 ± 0.0160
n=100 samples=0.5 → Train: 0.9858 | CV: 0.8020 ± 0.0140
n=100 samples=0.7 → Train: 0.9988 | CV: 0.8080 ± 0.0325
n=100 samples=1.0 → Train: 1.0000 | CV: 0.8040 ± 0.0193
n=200 samples=0.5 → Train: 0.9875 | CV: 0.8030 ± 0.0169
n=200 samples=0.7 → Train: 0.9998 | CV: 0.8050 ± 0.0197
n=200 samples=1.0 → Train: 1.0000 | CV: 0.8070 ± 0.0199


A partir de ~100 estimadores el rendimiento se estabiliza.
Un valor intermedio de max_samples (≈0.7) suele ofrecer mejor equilibrio entre diversidad y estabilidad.

Fijamos n_estimators=100 y max_samples=0.7 (los mejores del ejercicio anterior) y ahora exploramos el efecto de max_features: cuántas features ve cada árbol, probando desde el 10% hasta la raíz cuadrada del total. El objetivo es ver cuánta aleatoriedad en las features beneficia al modelo.


In [54]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
import numpy as np

n_features = X.shape[1]  

for max_features in [0.1, 0.2, 0.3, 0.5, int(np.sqrt(n_features))]:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        max_samples=0.7,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(bag, X, y)
    print(f"feat={max_features} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

feat=0.1 → Train: 1.0000 | CV: 0.8160 ± 0.0073
feat=0.2 → Train: 0.9995 | CV: 0.8110 ± 0.0183
feat=0.3 → Train: 0.9998 | CV: 0.8160 ± 0.0206
feat=0.5 → Train: 0.9995 | CV: 0.8060 ± 0.0150
feat=24 → Train: 1.0000 | CV: 0.8030 ± 0.0322


Fijamos los mejores valores hasta ahora (n_estimators=100, max_samples=0.7, max_features=0.1) e introducimos max_depth para controlar la complejidad del árbol base. El objetivo es encontrar el equilibrio entre un árbol demasiado simple (underfitting) y uno que memorice (overfitting).

In [55]:
for max_depth in [3, 5, 7, 10, 15, 20, 100]:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=max_depth, random_state=42),
        n_estimators=100,
        max_samples=0.7,
        max_features=0.1,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(bag, X, y)
    print(f"depth={max_depth} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

depth=3 → Train: 0.8645 | CV: 0.8070 ± 0.0209
depth=5 → Train: 0.9585 | CV: 0.8180 ± 0.0129
depth=7 → Train: 0.9902 | CV: 0.8180 ± 0.0112
depth=10 → Train: 0.9972 | CV: 0.8140 ± 0.0111
depth=15 → Train: 1.0000 | CV: 0.8200 ± 0.0045
depth=20 → Train: 1.0000 | CV: 0.8160 ± 0.0073
depth=100 → Train: 1.0000 | CV: 0.8160 ± 0.0073


### AFINAMOS - decision tree

AFINAMOS MAX SAMPLES

In [56]:
for max_samples in [0.6, 0.65, 0.7, 0.75, 0.8]:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=15, random_state=42),
        n_estimators=100,
        max_samples=max_samples,
        max_features=0.1,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(bag, X, y)
    print(f"samples={max_samples} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

samples=0.6 → Train: 0.9992 | CV: 0.8190 ± 0.0169
samples=0.65 → Train: 0.9995 | CV: 0.8230 ± 0.0075
samples=0.7 → Train: 1.0000 | CV: 0.8200 ± 0.0045
samples=0.75 → Train: 1.0000 | CV: 0.8190 ± 0.0206
samples=0.8 → Train: 1.0000 | CV: 0.8120 ± 0.0154


In [57]:
for max_samples in [0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68]:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=15, random_state=42),
        n_estimators=100,
        max_samples=max_samples,
        max_features=0.1,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(bag, X, y)
    print(f"samples={max_samples} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

samples=0.62 → Train: 0.9992 | CV: 0.8160 ± 0.0271
samples=0.63 → Train: 0.9990 | CV: 0.8120 ± 0.0223
samples=0.64 → Train: 0.9995 | CV: 0.8060 ± 0.0213
samples=0.65 → Train: 0.9995 | CV: 0.8230 ± 0.0075
samples=0.66 → Train: 0.9995 | CV: 0.8220 ± 0.0194
samples=0.67 → Train: 0.9998 | CV: 0.8150 ± 0.0187
samples=0.68 → Train: 1.0000 | CV: 0.8140 ± 0.0171


AFINAMOS MAX FEATURES

In [58]:
for max_features in [0.05, 0.08, 0.1, 0.12, 0.15]:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=15, random_state=42),
        n_estimators=100,
        max_samples=0.65,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(bag, X, y)
    print(f"feat={max_features} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

feat=0.05 → Train: 0.9998 | CV: 0.8060 ± 0.0159
feat=0.08 → Train: 0.9995 | CV: 0.8210 ± 0.0136
feat=0.1 → Train: 0.9995 | CV: 0.8230 ± 0.0075
feat=0.12 → Train: 1.0000 | CV: 0.8150 ± 0.0239
feat=0.15 → Train: 0.9990 | CV: 0.8200 ± 0.0095


AFINAMOS MAX_DEPTH

In [59]:
depths = [3, 5, 7, 8, 9, 10, 12, 15, 20, None]

results = []
for d in depths:
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=d, random_state=42),
        n_estimators=100,
        max_samples=0.65,
        max_features=0.1,
        random_state=42,
        n_jobs=-1
    )
    mean, std, train = evaluate(model, X, y)
    results.append((d, train, mean, std))
    label = str(d) if d is not None else "None"
    print(f"depth={label:>4} → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

depth=   3 → Train: 0.8672 | CV: 0.8080 ± 0.0286
depth=   5 → Train: 0.9570 | CV: 0.8160 ± 0.0159
depth=   7 → Train: 0.9890 | CV: 0.8180 ± 0.0136
depth=   8 → Train: 0.9918 | CV: 0.8150 ± 0.0145
depth=   9 → Train: 0.9945 | CV: 0.8140 ± 0.0107
depth=  10 → Train: 0.9972 | CV: 0.8100 ± 0.0130
depth=  12 → Train: 0.9988 | CV: 0.8170 ± 0.0129
depth=  15 → Train: 0.9995 | CV: 0.8230 ± 0.0075
depth=  20 → Train: 0.9995 | CV: 0.8210 ± 0.0120
depth=None → Train: 0.9995 | CV: 0.8210 ± 0.0120


### MODELO FINAL - decision tree

In [60]:
bagging_final = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=15, random_state=42),
    n_estimators=100,
    max_samples=0.65,
    max_features=0.1,
    random_state=42,
    n_jobs=-1
)

mean, std, train = evaluate(bagging_final, X, y)
print(f"Bagging final → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

Bagging final → Train: 0.9995 | CV: 0.8230 ± 0.0075


### LDA

El modelo LDA final ya presenta un rendimiento estable con una baja varianza, por lo que la aplicación de bagging no produce mejoras significativas en precisión. Esto se debe a que LDA es un modelo intrínsecamente estable, donde la agregación de múltiples estimadores aporta únicamente una reducción marginal de la varianza.

In [61]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif

lda_final = Pipeline([
    ("scaler",   PowerTransformer()),
    ("selector", SelectKBest(score_func=mutual_info_classif, k=20)),
    ("lda",      LinearDiscriminantAnalysis(solver="lsqr", shrinkage=0.3))
])

mean, std, train = evaluate(lda_final, X, y)
print(f"LDA final → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

LDA final → Train: 0.7642 | CV: 0.7630 ± 0.0238


In [62]:
from sklearn.ensemble import BaggingClassifier

bagging_lda = BaggingClassifier(
    estimator=lda_final,   # tu pipeline LDA completo
    n_estimators=100,
    max_samples=0.7,
    max_features=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

mean, std, train = evaluate(bagging_lda, X, y)

print(f"Bagging + LDA → Train: {train:.4f} | CV: {mean:.4f} ± {std:.4f}")

Bagging + LDA → Train: 0.7685 | CV: 0.7630 ± 0.0262


In [64]:
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import StratifiedKFold

# 🔥 IMPORTANTE: no usar cv como variable numérica
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for n in [50, 100, 200, 300]:
    for s in [0.5, 0.7, 1.0]:
        for f in [0.3, 0.5, 1.0]:

            model = BaggingClassifier(
                estimator=lda_final,   # tu LDA optimizado
                n_estimators=n,
                max_samples=s,
                max_features=f,
                bootstrap=True,
                random_state=42,
                n_jobs=-1
            )

            scores = cross_validate(
                model,
                X,
                y,
                cv=cv_strategy,
                scoring="accuracy",
                return_train_score=True
            )

            train = scores["train_score"].mean()
            cv_mean = scores["test_score"].mean()
            cv_std = scores["test_score"].std()

            results.append([n, s, f, train, cv_mean, cv_std])

            print(f"n={n}, s={s}, f={f} → CV={cv_mean:.4f} ± {cv_std:.4f}")

n=50, s=0.5, f=0.3 → CV=0.7670 ± 0.0236
n=50, s=0.5, f=0.5 → CV=0.7620 ± 0.0248
n=50, s=0.5, f=1.0 → CV=0.7690 ± 0.0193
n=50, s=0.7, f=0.3 → CV=0.7630 ± 0.0254
n=50, s=0.7, f=0.5 → CV=0.7630 ± 0.0250
n=50, s=0.7, f=1.0 → CV=0.7660 ± 0.0265
n=50, s=1.0, f=0.3 → CV=0.7640 ± 0.0263
n=50, s=1.0, f=0.5 → CV=0.7610 ± 0.0280
n=50, s=1.0, f=1.0 → CV=0.7650 ± 0.0217
n=100, s=0.5, f=0.3 → CV=0.7640 ± 0.0317
n=100, s=0.5, f=0.5 → CV=0.7670 ± 0.0294
n=100, s=0.5, f=1.0 → CV=0.7660 ± 0.0203
n=100, s=0.7, f=0.3 → CV=0.7620 ± 0.0289
n=100, s=0.7, f=0.5 → CV=0.7700 ± 0.0310
n=100, s=0.7, f=1.0 → CV=0.7630 ± 0.0262
n=100, s=1.0, f=0.3 → CV=0.7610 ± 0.0282
n=100, s=1.0, f=0.5 → CV=0.7670 ± 0.0250
n=100, s=1.0, f=1.0 → CV=0.7640 ± 0.0203
n=200, s=0.5, f=0.3 → CV=0.7620 ± 0.0314
n=200, s=0.5, f=0.5 → CV=0.7640 ± 0.0262


KeyboardInterrupt: 